# Gemini API Introduction

This notebook demonstrates how to interact with the **Gemini API** using direct HTTP requests.

## What You'll Learn

* **Setup & Authentication**: Use notebook widgets for secure credential input
* **Model Discovery**: Query and list available Gemini models
* **Simple API Calls**: Make single-turn requests to the API
* **Multi-Turn Conversations**: Maintain conversation context across multiple exchanges
* **Interactive Conversations**: Build real chatbot-style interactions where responses drive the next request
* **Document Understanding**: Extract text from .docx files using standard library (no external packages) and analyze with Gemini

## Technical Approach

This notebook uses Python's `requests` library for direct HTTP calls to the Gemini API. This gives you:
* Full control over request/response handling
* No dependency conflicts
* Clear visibility into API behavior

**All examples work with standard libraries** - Example 3 (document understanding) extracts text from .docx files using only `zipfile` and `xml.etree.ElementTree`, requiring no external packages beyond what's pre-installed in Databricks Runtime.

---

## 📋 Getting Started

**Run the cells in order:**
1. Create credential input widgets (Cell 2)
2. **Enter your API credentials** in the widgets at the top of the notebook
3. Load credentials and query available models (Cell 3)
4. Select a model (Cell 4)
5. Try the examples:
   - **Example 1** (Cells 5-6): Simple API call
   - **Example 2** (Cells 7-8): Interactive multi-turn conversation
   - **Example 3** (Cells 9-12): Document understanding with S3

---

In [0]:
# ==========================================
# CREATE CREDENTIAL INPUT WIDGETS
# ==========================================
# Notebook widgets provide a secure way to input credentials without
# embedding them directly in code. They appear at the top of the notebook
# as text input boxes.
#
# BENEFITS:
# - No credentials stored in notebook code
# - Easy to update without editing cells
# - Can be parameterized for different users/environments
# ==========================================

# Remove any existing widgets to start fresh
dbutils.widgets.removeAll()

# Create three text input widgets:
# 1. API Key (required) - your Gemini API authentication key
# 2. Base URL (pre-filled) - the API endpoint
# 3. Username (optional) - for tracking/logging purposes
dbutils.widgets.text("api_key", "", "API Key")
dbutils.widgets.text("base_url", "https://api.genai.mil/v1", "Base URL")
dbutils.widgets.text("username", "", "Username")

print("✅ Input widgets created successfully!")
print("")
print("🔑 NEXT STEP: Look at the top of this notebook and fill in:")
print("   1. Your API Key (required)")
print("   2. Verify the Base URL (pre-filled)")
print("   3. Add your username")
print("")
print("Then proceed to Cell 5 to load credentials and connect to the API.")

✅ Input widgets created successfully!

🔑 NEXT STEP: Look at the top of this notebook and fill in:
   1. Your API Key (required)
   2. Verify the Base URL (pre-filled)
   3. Optionally add your username

Then proceed to Cell 5 to load credentials and connect to the API.


In [0]:
# ==========================================
# TECHNICAL NOTE
# ==========================================
# This notebook uses Python's 'requests' library for direct HTTP calls
# rather than the OpenAI Python SDK. This approach:
# - Avoids dependency conflicts (OpenAI SDK requires pydantic v2)
# - Works with standard libraries available on this cluster
# - Provides full transparency into API request/response structure
# ==========================================

import json
import requests

# ==========================================
# LOAD CREDENTIALS FROM WIDGETS
# ==========================================
print("Loading credentials from widgets...")

# Read credentials from notebook widgets (input boxes at the top of the notebook)
api_key = dbutils.widgets.get("api_key")
base_url = dbutils.widgets.get("base_url")
username = dbutils.widgets.get("username")

# Validate required fields
if not api_key:
    raise ValueError("❌ API Key is required. Please enter your API key in the 'API Key' widget at the top of the notebook.")

credentials = {
    "api_key": api_key,
    "base_url": base_url,
    "username": username if username else "(not provided)"
}

print(f"✅ Credentials loaded for: {credentials['username']}")

# ==========================================
# INITIALIZE HTTP CLIENT
# ==========================================
base_url = credentials['base_url']
api_key = credentials['api_key']
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# ==========================================
# QUERY AVAILABLE MODELS
# ==========================================
print("\nQuerying available models...")
try:
    response = requests.get(f"{base_url}/models", headers=headers)
    response.raise_for_status()
    models_data = response.json()
    
    print("\n✅ Available Models:")
    for model in models_data.get('data', []):
        print(f"   • {model.get('id', 'unknown')}")
        
except Exception as e:
    print(f"❌ Failed to retrieve models: {e}")

Loading credentials from widgets...
✅ Credentials loaded for: (Your Email)

Querying available models...

✅ Available Models:
   • gemini-2.5-pro
   • gemini-3.1-pro-preview
   • gemini-3.5-flash
   • gemini-3.6-flash
   • gemini-3.7-flash
   • gemini-3.8-flash


In [0]:
# ==========================================
# MODEL SELECTION
# ==========================================
# Choose which Gemini model to use for the examples below.
# Different models have different capabilities and performance:
#
# - gemini-X.X-flash: Faster responses, good for most tasks
# - gemini-X.X-pro: More capable, better for complex reasoning
#
# See the list of available models from Cell 5 output above.
# ==========================================

# Set your chosen model here:
SELECTED_MODEL = "gemini-3.7-flash"

print(f"✅ Selected model: {SELECTED_MODEL}")
print("")
print("💡 TIP: You can change this to any model from the list above and re-run the examples.")

✅ Selected model: gemini-3.7-flash

💡 TIP: You can change this to any model from the list above and re-run the examples.


---

# 💬 Example 1: Simple API Call

This example demonstrates the **basic pattern** for making a single API request:

1. **Define a prompt** (user message)
2. **Construct the request payload** with model, messages, and parameters
3. **Send HTTP POST** to the `/chat/completions` endpoint
4. **Extract and display** the response

**Key Parameters:**
* `temperature`: Controls randomness (0.0 = deterministic, 1.0 = creative)
* `max_tokens`: Limits response length

---

In [0]:
# ==========================================
# SIMPLE API CALL EXAMPLE
# ==========================================

# Define a simple prompt
prompt = "Explain what the Gemini API is in 2-3 sentences."


import re

def render_markdown_to_html(text):
    """Convert common markdown patterns to clean HTML"""
    html = text.strip()
    
    # Helper to process inline markdown (bold, italic) in text
    def process_inline_markdown(text):
        # Handle bold **text**
        text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
        # Handle italic *text*
        text = re.sub(r'\*(.+?)\*', r'<em>\1</em>', text)
        return text
    
    # First pass: extract and replace tables with placeholders
    tables = []
    table_placeholder = "___TABLE_PLACEHOLDER_{}___"
    
    lines = html.split('\n')
    in_table = False
    current_table = []
    processed_lines = []
    
    for i, line in enumerate(lines):
        if '|' in line and not re.match(r'^\s*\|?[-:\s|]+\|?\s*$', line):
            if not in_table:
                in_table = True
                current_table = ['<table style="width: 100%; border-collapse: collapse; margin: 15px 0;">']
                if i + 1 < len(lines) and re.match(r'^\s*\|?[-:\s|]+\|?\s*$', lines[i + 1]):
                    current_table.append('<thead><tr>')
                    cols = [col.strip() for col in line.split('|') if col.strip()]
                    for col in cols:
                        col_html = process_inline_markdown(col)
                        current_table.append(f'<th style="border: 1px solid #e0e0e0; padding: 8px; text-align: left; background: #f5f5f5;">{col_html}</th>')
                    current_table.append('</tr></thead><tbody>')
                else:
                    current_table.append('<tr>')
                    cols = [col.strip() for col in line.split('|') if col.strip()]
                    for col in cols:
                        col_html = process_inline_markdown(col)
                        current_table.append(f'<td style="border: 1px solid #e0e0e0; padding: 8px;">{col_html}</td>')
                    current_table.append('</tr>')
            else:
                current_table.append('<tr>')
                cols = [col.strip() for col in line.split('|') if col.strip()]
                for col in cols:
                    col_html = process_inline_markdown(col)
                    current_table.append(f'<td style="border: 1px solid #e0e0e0; padding: 8px;">{col_html}</td>')
                current_table.append('</tr>')
        elif re.match(r'^\s*\|?[-:\s|]+\|?\s*$', line):
            continue  # Skip separator rows
        else:
            if in_table:
                current_table.append('</tbody></table>')
                table_html = ''.join(current_table)
                tables.append(table_html)
                processed_lines.append(table_placeholder.format(len(tables) - 1))
                current_table = []
                in_table = False
            processed_lines.append(line)
    
    if in_table:
        current_table.append('</tbody></table>')
        table_html = ''.join(current_table)
        tables.append(table_html)
        processed_lines.append(table_placeholder.format(len(tables) - 1))
    
    html = '\n'.join(processed_lines)
    
    # Now process the rest of the markdown
    # Handle headers
    html = re.sub(r'^### (.+)$', r'<h3 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.1em;">\1</h3>', html, flags=re.MULTILINE)
    html = re.sub(r'^## (.+)$', r'<h2 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.2em;">\1</h2>', html, flags=re.MULTILINE)
    html = re.sub(r'^# (.+)$', r'<h1 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.3em;">\1</h1>', html, flags=re.MULTILINE)
    
    # Handle bold and italic
    html = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', html)
    html = re.sub(r'\*(.+?)\*', r'<em>\1</em>', html)
    
    # Handle horizontal rules
    html = re.sub(r'^---+$', '<hr style="border: none; border-top: 1px solid #e0e0e0; margin: 10px 0;">', html, flags=re.MULTILINE)
    
    # Handle paragraphs (double line breaks)
    paragraphs = html.split('\n\n')
    formatted_paragraphs = []
    for para in paragraphs:
        if para.strip():
            # Don't wrap if it's already an HTML element or table placeholder
            if para.strip().startswith('<') or '___TABLE_PLACEHOLDER_' in para:
                formatted_paragraphs.append(para)
            else:
                # Replace single line breaks with <br> within paragraphs
                para = para.replace('\n', '<br>')
                formatted_paragraphs.append(f'<div style="margin: 8px 0;">{para}</div>')
    
    html = ''.join(formatted_paragraphs)
    
    # Restore tables
    for i, table in enumerate(tables):
        html = html.replace(table_placeholder.format(i), table)
    
    return html



print(f"Sending prompt: '{prompt}'\n")

try:
    # Prepare the request payload
    payload = {
        "model": SELECTED_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
        "max_tokens": 500
    }
    
    # Make the API call
    response = requests.post(
        f"{base_url}/chat/completions",
        headers=headers,
        json=payload
    )
    response.raise_for_status()
    response_data = response.json()
    
    # Extract and render the response
    answer = response_data['choices'][0]['message']['content']
    answer_html = render_markdown_to_html(answer)
    
    # ==========================================
    # DISPLAY WITH CLEAN STYLING
    # ==========================================
    html_card = f"""
    <div style="max-width: 800px; margin: 20px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;">
        <!-- Header -->
        <div style="border-bottom: 2px solid #2196F3; padding-bottom: 15px; margin-bottom: 20px;">
            <h2 style="margin: 0; color: #333; font-size: 1.5em; font-weight: 600;">API Response</h2>
            <p style="margin: 5px 0 0 0; color: #666; font-size: 0.9em;">{prompt}</p>
        </div>
        
        <!-- Response -->
        <div style="background: #fafafa; padding: 20px; border-radius: 8px; border-left: 3px solid #2196F3;">
            <div style="background: white; padding: 20px; border-radius: 6px; 
                        border: 1px solid #e0e0e0; color: #333; line-height: 1.6;">
                {answer_html}
            </div>
        </div>
        
        <!-- Token Usage -->
        <div style="margin-top: 20px; padding: 15px; background: #f5f5f5; border-radius: 6px;">
            <div style="display: flex; justify-content: space-around; text-align: center;">
                <div>
                    <div style="font-size: 1.8em; font-weight: bold; color: #2196F3;">{response_data['usage']['prompt_tokens']}</div>
                    <div style="font-size: 0.85em; color: #666;">Prompt Tokens</div>
                </div>
                <div>
                    <div style="font-size: 1.8em; font-weight: bold; color: #2196F3;">{response_data['usage']['completion_tokens']}</div>
                    <div style="font-size: 0.85em; color: #666;">Completion Tokens</div>
                </div>
                <div>
                    <div style="font-size: 1.8em; font-weight: bold; color: #2196F3;">{response_data['usage']['total_tokens']}</div>
                    <div style="font-size: 0.85em; color: #666;">Total Tokens</div>
                </div>
            </div>
        </div>
    </div>
    """
    displayHTML(html_card)
    
    print("✅ API call completed successfully!")
    
except Exception as e:
    print(f"❌ API call failed: {e}")

Sending prompt: 'Explain what the Gemini API is in 2-3 sentences.'



<!-- Header -->
 
 API Response 
 Explain what the Gemini API is in 2-3 sentences. 
 
 
 <!-- Response -->
 
 
 🌐 Overview of the Gemini API The Gemini API is Google's developer platform that provides direct programmatic access to its multimodal foundation models. It enables applications to process, reason over, and generate diverse content spanning text, code, images, audio, and video via REST endpoints and client SDKs. Developers utilize it to build capabilities ranging from semantic search and structured extraction to function calling and agentic workflows. 
 
 
 
 <!-- Token Usage -->
 
 
 
 14 
 Prompt Tokens 
 
 
 88 
 Completion Tokens 
 
 
 102 
 Total Tokens

✅ API call completed successfully!


---

# 🤖 Example 2: True Interactive Multi-Turn Conversation

This example demonstrates how to build a **real chatbot** where:

1. 🗨️ **User asks a question**
2. 📤 **Send request to API** with current conversation history
3. 📥 **Capture the API response**
4. 📄 **Add both user message and assistant response to conversation history**
5. 🔁 **Repeat** - the next request includes the full conversation

**Key Concept:**
* Each assistant response comes from the **actual API** (not hard-coded)
* The conversation builds naturally through real interactions
* This is the core pattern for building conversational AI applications

**Current Demo:**
The code below demonstrates **4 turns** of conversation:
* Turn 1: "What is the capital of France?"
* Turn 2: "What is the population of that city?" (pronoun reference)
* Turn 3: "What's a famous landmark there?" (location reference)
* Turn 4: "When was it built?" (continuing context)

**📝 How to Add More Turns:**

To add additional conversation turns, simply copy this pattern before the display section:

```python
# ==========================================
# TURN 5: Your next question
# ==========================================
user_message_5 = "Your question here"
conversation.append({"role": "user", "content": user_message_5})

try:
    assistant_response_5 = make_api_call(conversation)
    conversation.append({"role": "assistant", "content": assistant_response_5})
except Exception as e:
    print(f"❌ Turn 5 failed: {e}")
    raise
```

**Tips for Adding Turns:**
* Use pronouns ("it", "that", "there") to test context retention
* Each turn automatically includes all previous messages
* The display section will automatically show all turns
* No limit on conversation length (but watch token usage)

---

In [0]:
# ==========================================
# TRUE INTERACTIVE MULTI-TURN CONVERSATION
# ==========================================
# This demonstrates how to build a real chatbot where each response
# feeds into the next request

import re

def render_markdown_to_html(text):
    """Convert common markdown patterns to clean HTML"""
    html = text.strip()
    
    # Helper to process inline markdown (bold, italic) in text
    def process_inline_markdown(text):
        # Handle bold **text**
        text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
        # Handle italic *text*
        text = re.sub(r'\*(.+?)\*', r'<em>\1</em>', text)
        return text
    
    # First pass: extract and replace tables with placeholders
    tables = []
    table_placeholder = "___TABLE_PLACEHOLDER_{}___"
    
    lines = html.split('\n')
    in_table = False
    current_table = []
    processed_lines = []
    
    for i, line in enumerate(lines):
        if '|' in line and not re.match(r'^\s*\|?[-:\s|]+\|?\s*$', line):
            if not in_table:
                in_table = True
                current_table = ['<table style="width: 100%; border-collapse: collapse; margin: 15px 0;">']
                if i + 1 < len(lines) and re.match(r'^\s*\|?[-:\s|]+\|?\s*$', lines[i + 1]):
                    current_table.append('<thead><tr>')
                    cols = [col.strip() for col in line.split('|') if col.strip()]
                    for col in cols:
                        col_html = process_inline_markdown(col)
                        current_table.append(f'<th style="border: 1px solid #e0e0e0; padding: 8px; text-align: left; background: #f5f5f5;">{col_html}</th>')
                    current_table.append('</tr></thead><tbody>')
                else:
                    current_table.append('<tr>')
                    cols = [col.strip() for col in line.split('|') if col.strip()]
                    for col in cols:
                        col_html = process_inline_markdown(col)
                        current_table.append(f'<td style="border: 1px solid #e0e0e0; padding: 8px;">{col_html}</td>')
                    current_table.append('</tr>')
            else:
                current_table.append('<tr>')
                cols = [col.strip() for col in line.split('|') if col.strip()]
                for col in cols:
                    col_html = process_inline_markdown(col)
                    current_table.append(f'<td style="border: 1px solid #e0e0e0; padding: 8px;">{col_html}</td>')
                current_table.append('</tr>')
        elif re.match(r'^\s*\|?[-:\s|]+\|?\s*$', line):
            continue  # Skip separator rows
        else:
            if in_table:
                current_table.append('</tbody></table>')
                table_html = ''.join(current_table)
                tables.append(table_html)
                processed_lines.append(table_placeholder.format(len(tables) - 1))
                current_table = []
                in_table = False
            processed_lines.append(line)
    
    if in_table:
        current_table.append('</tbody></table>')
        table_html = ''.join(current_table)
        tables.append(table_html)
        processed_lines.append(table_placeholder.format(len(tables) - 1))
    
    html = '\n'.join(processed_lines)
    
    # Now process the rest of the markdown
    # Handle headers
    html = re.sub(r'^### (.+)$', r'<h3 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.1em;">\1</h3>', html, flags=re.MULTILINE)
    html = re.sub(r'^## (.+)$', r'<h2 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.2em;">\1</h2>', html, flags=re.MULTILINE)
    html = re.sub(r'^# (.+)$', r'<h1 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.3em;">\1</h1>', html, flags=re.MULTILINE)
    
    # Handle bold and italic
    html = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', html)
    html = re.sub(r'\*(.+?)\*', r'<em>\1</em>', html)
    
    # Handle horizontal rules
    html = re.sub(r'^---+$', '<hr style="border: none; border-top: 1px solid #e0e0e0; margin: 10px 0;">', html, flags=re.MULTILINE)
    
    # Handle paragraphs (double line breaks)
    paragraphs = html.split('\n\n')
    formatted_paragraphs = []
    for para in paragraphs:
        if para.strip():
            # Don't wrap if it's already an HTML element or table placeholder
            if para.strip().startswith('<') or '___TABLE_PLACEHOLDER_' in para:
                formatted_paragraphs.append(para)
            else:
                # Replace single line breaks with <br> within paragraphs
                para = para.replace('\n', '<br>')
                formatted_paragraphs.append(f'<div style="margin: 8px 0;">{para}</div>')
    
    html = ''.join(formatted_paragraphs)
    
    # Restore tables
    for i, table in enumerate(tables):
        html = html.replace(table_placeholder.format(i), table)
    
    return html

# Initialize conversation history
conversation = []

def make_api_call(conversation_history):
    """Helper function to make API calls and return the response"""
    payload = {
        "model": SELECTED_MODEL,
        "messages": conversation_history,
        "temperature": 0.7
    }
    
    response = requests.post(
        f"{base_url}/chat/completions",
        headers=headers,
        json=payload
    )
    response.raise_for_status()
    response_data = response.json()
    
    return response_data['choices'][0]['message']['content']

print("Starting conversation demonstration...\n")

# ==========================================
# TURN 1: First question
# ==========================================
user_message_1 = "What is the capital of France?"
conversation.append({"role": "user", "content": user_message_1})

try:
    assistant_response_1 = make_api_call(conversation)
    conversation.append({"role": "assistant", "content": assistant_response_1})
except Exception as e:
    print(f"❌ Turn 1 failed: {e}")
    raise

# ==========================================
# TURN 2: Follow-up using pronoun reference
# ==========================================
user_message_2 = "What is the population of that city?"
conversation.append({"role": "user", "content": user_message_2})

try:
    assistant_response_2 = make_api_call(conversation)
    conversation.append({"role": "assistant", "content": assistant_response_2})
except Exception as e:
    print(f"❌ Turn 2 failed: {e}")
    raise

# ==========================================
# TURN 3: Another follow-up
# ==========================================
user_message_3 = "What's a famous landmark there?"
conversation.append({"role": "user", "content": user_message_3})

try:
    assistant_response_3 = make_api_call(conversation)
    conversation.append({"role": "assistant", "content": assistant_response_3})
except Exception as e:
    print(f"❌ Turn 3 failed: {e}")
    raise

# ==========================================
# TURN 4: Continue the conversation
# ==========================================
user_message_4 = "When was it built?"
conversation.append({"role": "user", "content": user_message_4})

try:
    assistant_response_4 = make_api_call(conversation)
    conversation.append({"role": "assistant", "content": assistant_response_4})
except Exception as e:
    print(f"❌ Turn 4 failed: {e}")
    raise

# ==========================================
# DISPLAY CONVERSATION WITH CLEAN STYLING
# ==========================================

# Build conversation HTML with clean, minimal design
conversation_html = ""
for i in range(0, len(conversation), 2):
    turn_num = (i // 2) + 1
    user_msg = conversation[i]['content']
    assistant_msg = conversation[i + 1]['content'] if i + 1 < len(conversation) else ""
    
    # Render markdown in messages
    user_html = render_markdown_to_html(user_msg)
    assistant_html = render_markdown_to_html(assistant_msg) if assistant_msg else ""
    
    conversation_html += f"""
    <div style="margin: 30px 0; padding: 20px; background: #fafafa; border-radius: 8px; border-left: 3px solid #2196F3;">
        <!-- Turn header -->
        <div style="font-size: 0.85em; color: #666; margin-bottom: 15px; font-weight: 500;">
            Turn {turn_num}
        </div>
        
        <!-- User message -->
        <div style="margin-bottom: 15px;">
            <div style="font-size: 0.9em; color: #555; margin-bottom: 8px; font-weight: 600;">
                You
            </div>
            <div style="background: white; padding: 15px; border-radius: 6px; 
                        border: 1px solid #e0e0e0; color: #333; line-height: 1.6;">
                {user_html}
            </div>
        </div>
        
        <!-- Assistant response -->
        <div>
            <div style="font-size: 0.9em; color: #555; margin-bottom: 8px; font-weight: 600;">
                Gemini
            </div>
            <div style="background: white; padding: 15px; border-radius: 6px; 
                        border: 1px solid #e0e0e0; color: #333; line-height: 1.6;">
                {assistant_html}
            </div>
        </div>
    </div>
    """

# Wrap in container
full_html = f"""
<div style="max-width: 800px; margin: 20px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;">
    <!-- Header -->
    <div style="border-bottom: 2px solid #2196F3; padding-bottom: 15px; margin-bottom: 20px;">
        <h2 style="margin: 0; color: #333; font-size: 1.5em; font-weight: 600;">Multi-Turn Conversation</h2>
        <p style="margin: 5px 0 0 0; color: #666; font-size: 0.95em;">Demonstrating context retention across {len(conversation) // 2} turns</p>
    </div>
    
    <!-- Conversation -->
    {conversation_html}
    
    <!-- Footer note -->
    <div style="margin-top: 30px; padding: 15px; background: #f5f5f5; border-radius: 6px; 
                font-size: 0.9em; color: #555; line-height: 1.6;">
        <strong>💡 Key Concept:</strong> Each API request includes the complete conversation history. 
        Notice how the assistant understands "that city" (Turn 2) and "there" (Turn 3) 
        by referencing the context from Turn 1.
    </div>
</div>
"""

displayHTML(full_html)

print(f"✅ Conversation complete! Total turns: {len(conversation) // 2}")
print(f"📊 Total messages in history: {len(conversation)}")

Starting conversation demonstration...



<!-- Header -->
 
 Multi-Turn Conversation 
 Demonstrating context retention across 4 turns 
 
 
 <!-- Conversation -->
 
 
 <!-- Turn header -->
 
 Turn 1
 
 
 <!-- User message -->
 
 
 You
 
 
 What is the capital of France? 
 
 
 
 <!-- Assistant response -->
 
 
 Gemini
 
 
 The capital of France is Paris . 
 📍 Paris Overview & Population Metric Details Capital of France City Proper Population ~2.1 million (covering the 20 central arrondissements) Metropolitan Area Population ~13.1 million (the entire urban and commuter belt) Region Île-de-France Famous Landmarks Eiffel Tower, Louvre Museum, Notre-Dame Cathedral 
 
 
 
 
 
 <!-- Turn header -->
 
 Turn 2
 
 
 <!-- User message -->
 
 
 You
 
 
 What is the population of that city? 
 
 
 
 <!-- Assistant response -->
 
 
 Gemini
 
 
 👥 Population of Paris Scope Estimated Population Description City Proper ~2.1 million The administrative municipality (the 20 central arrondissements) Urban Unit ( Unité urbaine ) ~10.9 million The continuously built-up urban area Metropolitan Area ( Aire d'attraction ) ~13.1 million The broader commuter and metropolitan region 

 
 
 
 
 
 <!-- Turn header -->
 
 Turn 3
 
 
 <!-- User message -->
 
 
 You
 
 
 What's a famous landmark there? 
 
 
 
 <!-- Assistant response -->
 
 
 Gemini
 
 
 The most iconic landmark in Paris is the Eiffel Tower ( La Tour Eiffel ). Completed in 1889 for the Exposition Universelle (World's Fair), it stands 330 meters (1,083 feet) tall along the Seine River and is one of the most visited paid monuments in the world. 
 🏛️ Famous Landmarks in Paris Landmark Highlights Built / Established Eiffel Tower Iconic wrought-iron lattice tower offering panoramic city views 1889 Louvre Museum World's largest art museum, home to the Mona Lisa 1793 Notre-Dame Cathedral Masterpiece of French Gothic architecture on the Île de la Cité 1163–1345 Arc de Triomphe Monument honoring those who fought and died for France 1836 Sacré-Cœur Basilica Romano-Byzantine basilica atop the Montmartre hill 1914 
 
 
 
 
 
 <!-- Turn header -->
 
 Turn 4
 
 
 <!-- User message -->
 
 
 You
 
 
 When was it built? 
 
 
 
 <!-- Assistant response -->
 
 
 Gemini
 
 
 The Eiffel Tower was built between January 28, 1887, and March 31, 1889 . It was constructed as the centerpiece and entrance arch for the 1889 Exposition Universelle (World's Fair) to celebrate the centennial of the French Revolution. 
 ⏱️ Eiffel Tower Construction Timeline Milestone Date / Timeline Details Construction Began January 28, 1887 Foundation work started along the Champ de Mars Pillar Assembly July 1887 – December 1887 Construction of the four main lattice legs First Platform Reached April 1, 1888 Joined the four pillars into a unified base Structure Completed March 31, 1889 Gustave Eiffel raised the French flag at the top Public Opening May 6, 1889 Officially opened to visitors at the World's Fair Total Build Time 2 years, 2 months, 5 days Remarkably fast given the scale and technology 
 🏛️ Construction Dates of Other Paris Landmarks Landmark Construction / Opening Dates Notre-Dame Cathedral 1163 – 1345 Arc de Triomphe 1806 – 1836 Louvre Palace / Museum Built as a fortress in 1190; opened as a museum in 1793 Sacré-Cœur Basilica 1875 – 1914 
 
 
 
 
 
 <!-- Footer note -->
 
 💡 Key Concept: Each API request includes the complete conversation history. 
 Notice how the assistant understands "that city" (Turn 2) and "there" (Turn 3) 
 by referencing the context from Turn 1.

✅ Conversation complete! Total turns: 4
📊 Total messages in history: 8


---

# 📄 Example 3: Document Understanding with S3

This example demonstrates how to **analyze documents** using the Gemini API with **no external dependencies**:

1. 🗄️ **Connect to S3**: Access AWS S3 buckets from Databricks using boto3 (pre-installed)
2. 📥 **Download a document**: Retrieve a .docx file from S3
3. 📝 **Extract text**: Parse .docx files using **standard library only** (zipfile + XML parsing)
4. 📤 **Send to Gemini API**: Analyze the extracted text with Gemini
5. 🤔 **Ask multiple follow-up questions**: Automatically ask 3 customizable questions and display answers in clean HTML cards with proper formatting

**Key Innovations:**
* **No python-docx required!** Word .docx files are ZIP archives containing XML
* Uses only `zipfile` and `xml.etree.ElementTree` (both standard library)
* Works immediately on any Databricks cluster
* **Clean HTML visualization**: Q&A displayed in styled cards with properly rendered markdown tables, headers, and formatting

**Use Cases:**
* Document summarization and analysis
* Contract review and key information extraction
* Report understanding and Q&A
* Meeting notes processing
* Technical documentation queries

**Note:** This example extracts text from .docx files. For PDF or other formats, you would need additional libraries or pre-processing.

---

In [0]:
# ==========================================
# S3 CONNECTION AND FILE DOWNLOAD
# ==========================================
# Connect to AWS S3 and download a document for analysis

import boto3
import io
import base64
from botocore.exceptions import ClientError

# ==========================================
# S3 CONFIGURATION
# ==========================================
S3_BUCKET = "advana-restricted-data-zone"
S3_KEY = "builder/jup/swctc/edl/Gemini_API_Demo_Doc_20260831210229.docx"
S3_REGION = "us-gov-west-1"

print("Connecting to S3...")
print(f"Bucket: {S3_BUCKET}")
print(f"Key: {S3_KEY}")
print(f"Region: {S3_REGION}")
print("="*80)

try:
    # ==========================================
    # CREATE S3 CLIENT
    # ==========================================
    # Using default credentials from the Databricks cluster/IAM role
    s3_client = boto3.client('s3', region_name=S3_REGION)
    
    print("\n✅ S3 client created successfully")
    
    # ==========================================
    # DOWNLOAD FILE FROM S3
    # ==========================================
    print(f"\nDownloading file from S3...")
    
    # Download to in-memory buffer
    file_buffer = io.BytesIO()
    s3_client.download_fileobj(S3_BUCKET, S3_KEY, file_buffer)
    file_buffer.seek(0)  # Reset buffer position to beginning
    
    # Get file info
    file_bytes = file_buffer.read()
    file_size_kb = len(file_bytes) / 1024
    
    print(f"✅ File downloaded successfully!")
    print(f"   File size: {file_size_kb:.2f} KB")
    print(f"   File type: .docx (Word document)")
    
    # ==========================================
    # PREPARE FILE FOR API
    # ==========================================
    # Convert to base64 for API upload
    file_base64 = base64.b64encode(file_bytes).decode('utf-8')
    
    print(f"\n✅ File encoded for API upload")
    print(f"   Base64 length: {len(file_base64)} characters")
    
    # Store for next cell
    document_data = {
        'filename': S3_KEY.split('/')[-1],
        'content_base64': file_base64,
        'size_kb': file_size_kb,
        'mime_type': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'
    }
    
    print("\n" + "="*80)
    print("✅ Ready to analyze document with Gemini API!")
    print(f"   Document: {document_data['filename']}")
    print(f"   Size: {document_data['size_kb']:.2f} KB")
    
except ClientError as e:
    error_code = e.response['Error']['Code']
    error_message = e.response['Error']['Message']
    print(f"\n❌ S3 Error ({error_code}): {error_message}")
    print("\n🔑 Common issues:")
    print("   • Missing IAM permissions - ensure the cluster has S3 read access")
    print("   • Incorrect bucket name or key")
    print("   • Bucket in different region")
    raise
    
except Exception as e:
    print(f"\n❌ Unexpected error: {e}")
    raise

Connecting to S3...
Bucket: advana-restricted-data-zone
Key: builder/jup/swctc/edl/Gemini_API_Demo_Doc_20260831210229.docx
Region: us-gov-west-1

✅ S3 client created successfully

✅ File downloaded successfully!
   File size: 36.93 KB
   File type: .docx (Word document)

✅ File encoded for API upload
   Base64 length: 50424 characters

✅ Ready to analyze document with Gemini API!
   Document: Gemini_API_Demo_Doc_20260831210229.docx
   Size: 36.93 KB


In [0]:
# ==========================================
# EXTRACT TEXT AND ANALYZE DOCUMENT
# ==========================================
# Extract text from .docx using standard library (no external packages needed)
# .docx files are ZIP archives containing XML - we parse the main document XML

import zipfile
import xml.etree.ElementTree as ET
import base64
import io

if 'document_data' not in locals():
    print("⚠️  No document loaded!")
    print("Please run Cell 13 to download the document from S3.")
else:
    print(f"Analyzing document: {document_data['filename']}")
    print("="*80)
    
    try:
        # ==========================================
        # STEP 1: EXTRACT TEXT FROM DOCX
        # ==========================================
        print("\n📄 Extracting text from document...")
        print("   Using standard library (zipfile + XML parsing)")
        
        # Decode base64 back to bytes
        docx_bytes = base64.b64decode(document_data['content_base64'])
        file_buffer = io.BytesIO(docx_bytes)
        
        # Extract text using zipfile + XML parsing
        # .docx files are ZIP archives; the main document is in word/document.xml
        with zipfile.ZipFile(file_buffer, 'r') as docx_zip:
            # Read the main document XML
            xml_content = docx_zip.read('word/document.xml')
            
            # Parse the XML
            tree = ET.fromstring(xml_content)
            
            # The namespace for Word documents
            namespace = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
            
            # Extract all text elements
            paragraphs = []
            for paragraph in tree.findall('.//w:p', namespace):
                texts = []
                for text_element in paragraph.findall('.//w:t', namespace):
                    if text_element.text:
                        texts.append(text_element.text)
                if texts:
                    paragraphs.append(''.join(texts))
            
            # Join paragraphs
            document_text = '\n\n'.join(paragraphs)
        
        print(f"✅ Text extracted successfully!")
        print(f"   Total characters: {len(document_text):,}")
        print(f"   Total words: {len(document_text.split()):,}")
        
        # Show preview
        print(f"\n👁️  Document Preview (first 500 characters):")
        print("-"*80)
        print(document_text[:500] + "...")
        print("-"*80)
        
        # ==========================================
        # STEP 2: SEND TEXT TO GEMINI API FOR ANALYSIS
        # ==========================================
        print("\n🤖 Sending to Gemini API for analysis...\n")
        
        prompt = f"""Please analyze the following document and provide:
1. A brief summary of the main content
2. Key topics or themes discussed
3. Any important data, numbers, or findings mentioned

Document content:
---
{document_text}
---
"""
        
        payload = {
            "model": SELECTED_MODEL,
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 1500
        }
        
        response = requests.post(
            f"{base_url}/chat/completions",
            headers=headers,
            json=payload
        )
        response.raise_for_status()
        response_data = response.json()
        
        # Extract and display the analysis
        answer = response_data['choices'][0]['message']['content']
        
        print("✅ Document Analysis:")
        print("="*80)
        print(answer)
        print("="*80)
        
        # Display usage stats
        if 'usage' in response_data:
            print(f"\n📊 Token Usage:")
            print(f"   - Prompt tokens: {response_data['usage'].get('prompt_tokens', 'N/A')}")
            print(f"   - Completion tokens: {response_data['usage'].get('completion_tokens', 'N/A')}")
            print(f"   - Total tokens: {response_data['usage'].get('total_tokens', 'N/A')}")
        
        # Store document context for follow-up questions
        document_context = {
            'text': document_text,
            'filename': document_data['filename'],
            'char_count': len(document_text),
            'word_count': len(document_text.split())
        }
        
        print(f"\n💡 Document context stored! The next cell will automatically ask 3 follow-up questions.")
        print(f"   Questions cover: key recommendations, specific metrics, and document purpose.")
        print(f"   You can customize the questions by editing the 'follow_up_questions' list in Cell 12.")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
        raise

Analyzing document: Gemini_API_Demo_Doc_20260831210229.docx

📄 Extracting text from document...
   Using standard library (zipfile + XML parsing)
✅ Text extracted successfully!
   Total characters: 6,616
   Total words: 894

👁️  Document Preview (first 500 characters):
--------------------------------------------------------------------------------
Apex Global Technologies

Q3 Performance Review & Strategic Initiatives Report

Document ID: AGT-2024-Q3-REV

Publication Date: October 24, 2024

Author: Corporate Strategy & Business Intelligence Group

Target Audience: Executive Committee & Stakeholders

Classification: Internal Business Use

Table of Contents

Right-click and select 'Update Field' to populate the table of contents.

Executive Summary

During the third quarter of Fiscal Year 2024 (Q3 FY24), Apex Global Technologies demonstrate...
--------------------------------------------------------------------------------

🤖 Sending to Gemini API for analysis...

✅ Document Analysis:
#

In [0]:
# ==========================================
# FOLLOW-UP QUESTIONS ABOUT THE DOCUMENT
# ==========================================
# Now that we have the document context, we can ask multiple specific questions

import re

def render_markdown_to_html(text):
    """Convert common markdown patterns to clean HTML"""
    html = text.strip()
    
    # Helper to process inline markdown (bold, italic) in text
    def process_inline_markdown(text):
        # Handle bold **text**
        text = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', text)
        # Handle italic *text*
        text = re.sub(r'\*(.+?)\*', r'<em>\1</em>', text)
        return text
    
    # First pass: extract and replace tables with placeholders
    tables = []
    table_placeholder = "___TABLE_PLACEHOLDER_{}___"
    
    lines = html.split('\n')
    in_table = False
    current_table = []
    processed_lines = []
    
    for i, line in enumerate(lines):
        if '|' in line and not re.match(r'^\s*\|?[-:\s|]+\|?\s*$', line):
            if not in_table:
                in_table = True
                current_table = ['<table style="width: 100%; border-collapse: collapse; margin: 15px 0; font-size: 0.9em;">']
                if i + 1 < len(lines) and re.match(r'^\s*\|?[-:\s|]+\|?\s*$', lines[i + 1]):
                    current_table.append('<thead><tr>')
                    cols = [col.strip() for col in line.split('|') if col.strip()]
                    for col in cols:
                        col_html = process_inline_markdown(col)
                        current_table.append(f'<th style="border: 1px solid #e0e0e0; padding: 8px; text-align: left; background: #f5f5f5; font-weight: 600;">{col_html}</th>')
                    current_table.append('</tr></thead><tbody>')
                else:
                    current_table.append('<tr>')
                    cols = [col.strip() for col in line.split('|') if col.strip()]
                    for col in cols:
                        col_html = process_inline_markdown(col)
                        current_table.append(f'<td style="border: 1px solid #e0e0e0; padding: 8px;">{col_html}</td>')
                    current_table.append('</tr>')
            else:
                current_table.append('<tr>')
                cols = [col.strip() for col in line.split('|') if col.strip()]
                for col in cols:
                    col_html = process_inline_markdown(col)
                    current_table.append(f'<td style="border: 1px solid #e0e0e0; padding: 8px;">{col_html}</td>')
                current_table.append('</tr>')
        elif re.match(r'^\s*\|?[-:\s|]+\|?\s*$', line):
            continue  # Skip separator rows
        else:
            if in_table:
                current_table.append('</tbody></table>')
                table_html = ''.join(current_table)
                tables.append(table_html)
                processed_lines.append(table_placeholder.format(len(tables) - 1))
                current_table = []
                in_table = False
            processed_lines.append(line)
    
    if in_table:
        current_table.append('</tbody></table>')
        table_html = ''.join(current_table)
        tables.append(table_html)
        processed_lines.append(table_placeholder.format(len(tables) - 1))
    
    html = '\n'.join(processed_lines)
    
    # Now process the rest of the markdown
    # Handle headers
    html = re.sub(r'^### (.+)$', r'<h3 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.1em;">\1</h3>', html, flags=re.MULTILINE)
    html = re.sub(r'^## (.+)$', r'<h2 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.2em;">\1</h2>', html, flags=re.MULTILINE)
    html = re.sub(r'^# (.+)$', r'<h1 style="color: #333; margin: 15px 0 8px 0; font-weight: 600; font-size: 1.3em;">\1</h1>', html, flags=re.MULTILINE)
    
    # Handle bold and italic
    html = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', html)
    html = re.sub(r'\*(.+?)\*', r'<em>\1</em>', html)
    
    # Handle horizontal rules
    html = re.sub(r'^---+$', '<hr style="border: none; border-top: 1px solid #e0e0e0; margin: 15px 0;">', html, flags=re.MULTILINE)
    
    # Handle paragraphs
    paragraphs = html.split('\n\n')
    formatted_paragraphs = []
    for para in paragraphs:
        para = para.strip()
        if para:
            if para.startswith('<') or table_placeholder.split('_')[0] in para:
                formatted_paragraphs.append(para)
            else:
                para_with_breaks = para.replace('\n', '<br>')
                formatted_paragraphs.append(f'<div style="margin: 8px 0;">{para_with_breaks}</div>')
    
    html = ''.join(formatted_paragraphs)
    
    # Restore tables
    for i, table_html in enumerate(tables):
        html = html.replace(table_placeholder.format(i), table_html)
    
    return html

if 'document_context' not in locals():
    print("⚠️  No document context found!")
    print("Please run the text extraction cell first.")
else:
    print(f"Document: {document_context['filename']}")
    print(f"Context loaded: {document_context['word_count']:,} words")
    print("Asking multiple follow-up questions...\n")
    
    # ==========================================
    # EXAMPLE FOLLOW-UP QUESTIONS
    # ==========================================
    # You can customize these questions based on your document
    
    follow_up_questions = [
        "What are the key recommendations or action items in this document?",
        "Are there any specific numbers, dates, or metrics mentioned?",
        "What is the main purpose or goal of this document?"
    ]
    
    # Store Q&A pairs for display
    qa_pairs = []
    total_tokens = 0
    
    try:
        # Ask all questions
        for i, question in enumerate(follow_up_questions, 1):
            print(f"🔍 Question {i}/{len(follow_up_questions)}: {question[:60]}...")
            
            # Build the prompt with document context
            prompt = f"""Based on the following document, please answer this question:

Question: {question}

Document content:
---
{document_context['text']}
---

Please provide a clear and specific answer based only on the information in the document.
"""
            
            payload = {
                "model": SELECTED_MODEL,
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.7,
                "max_tokens": 1000
            }
            
            response = requests.post(
                f"{base_url}/chat/completions",
                headers=headers,
                json=payload
            )
            response.raise_for_status()
            response_data = response.json()
            
            # Extract the answer
            answer = response_data['choices'][0]['message']['content']
            tokens = response_data['usage'].get('total_tokens', 0)
            total_tokens += tokens
            
            # Convert markdown to HTML
            answer_html = render_markdown_to_html(answer)
            
            # Store Q&A pair
            qa_pairs.append({
                'question': question,
                'answer': answer_html,
                'tokens': tokens
            })
        
        print(f"\n✅ All questions answered!\n")
        
        # ==========================================
        # BUILD HTML DISPLAY
        # ==========================================
        html_sections = []
        
        for i, qa in enumerate(qa_pairs, 1):
            section = f"""
            <div style="margin: 20px 0; padding: 20px; background: #fafafa; border-radius: 8px; border-left: 3px solid #2196F3;">
                <div style="margin-bottom: 15px;">
                    <span style="display: inline-block; background: #2196F3; color: white; padding: 4px 12px; border-radius: 4px; font-weight: 600; font-size: 0.85em; margin-bottom: 8px;">Question {i}</span>
                    <h3 style="margin: 8px 0 0 0; color: #333; font-size: 1.05em; font-weight: 600; line-height: 1.4;">{qa['question']}</h3>
                </div>
                <div style="background: white; padding: 15px; border-radius: 6px; border: 1px solid #e0e0e0;">
                    <div style="color: #555; line-height: 1.6;">{qa['answer']}</div>
                </div>
            </div>
            """
            html_sections.append(section)
        
        # Build complete HTML
        html_output = f"""
        <div style="max-width: 1000px; margin: 20px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;">
            
            <!-- Header -->
            <div style="border-bottom: 2px solid #2196F3; padding-bottom: 15px; margin-bottom: 30px;">
                <h2 style="margin: 0; color: #333; font-size: 1.5em; font-weight: 600;">📄 Document Q&A</h2>
                <p style="margin: 5px 0 0 0; color: #666; font-size: 0.95em;">{document_context['filename']} • {len(follow_up_questions)} questions answered</p>
            </div>
            
            <!-- Q&A Sections -->
            {''.join(html_sections)}
            
            <!-- Footer Stats -->
            <div style="margin-top: 30px; padding: 15px; background: #f5f5f5; border-radius: 6px; text-align: center;">
                <div style="color: #666; font-size: 0.9em;">
                    <strong>Total Token Usage:</strong> <span style="color: #2196F3; font-weight: 600;">{total_tokens:,}</span> tokens
                </div>
            </div>
            
        </div>
        """
        
        displayHTML(html_output)
        
        print(f"📊 Summary: {len(qa_pairs)} questions answered, {total_tokens:,} total tokens used")
        print(f"\n💡 To ask different questions, modify the 'follow_up_questions' list in this cell and run again!")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
        raise

Document: Gemini_API_Demo_Doc_20260831210229.docx
Context loaded: 894 words
Asking multiple follow-up questions...

🔍 Question 1/3: What are the key recommendations or action items in this doc...
🔍 Question 2/3: Are there any specific numbers, dates, or metrics mentioned?...
🔍 Question 3/3: What is the main purpose or goal of this document?...

✅ All questions answered!



<!-- Header -->
 
 📄 Document Q&A 
 Gemini_API_Demo_Doc_20260831210229.docx • 3 questions answered 
 
 
 <!-- Q&A Sections -->
 
 
 
 Question 1 
 What are the key recommendations or action items in this document? 
 
 
 📌 Recommended Action Items (Q4 FY24) Based on the document, the four key recommendations and strategic action items are: Initiative Specific Action / Recommendation Target / Objective Enterprise AI General Availability Ensure final security validations for Project ApexAI are concluded prior to the annual Global Summit. Complete prior to November 20, 2024 . Consulting Margin Optimization Reallocate underutilized professional services headcount toward AI implementation advisory. Rebound Consulting gross margin to 40% by Q1 FY25 . CapEx Reallocation Authorize an additional $3.5M in compute hardware reserves. Support anticipated surge in Q4 multi-tenant API traffic. APAC Go-to-Market Alignment Schedule a dedicated executive committee review with MD APAC. Recalibrate local partner tiering and currency hedging strategies. 
 
 
 
 
 
 Question 2 
 Are there any specific numbers, dates, or metrics mentioned? 
 
 
 ```suggested_questions
- What is the significance of the October 24, 2024 publication date?
- Are there any dependencies between the Project ApexAI and Project Sentinel target dates?
- What are the specific KPIs being tracked for Cloud Migration 3.0?
```Yes, the document contains numerous specific numbers , dates , and metrics across financial, operational, and strategic project categories. 📅 Key Dates Mentioned Date / Timeline Context / Milestone October 24, 2024 Publication Date of the Report October 30, 2024 Project Sentinel completion date November 15, 2024 Project ApexAI target launch date November 20, 2024 Annual Global Summit December 31, 2024 Cloud Migration 3.0 target completion date February 28, 2025 Global Expansion (APAC) target setup date Q1 FY25 Target timeline to rebound Consulting gross margin to 40% 💵 Consolidated Financial & Expense Metrics Metric / Line Item Q3 FY24 Figure Context / Comparison Consolidated Revenue $148.5M +14.2% YoY vs. Q3 FY23 ($130.0M), +$4.5M (+3.1%) vs. Budget ($144.0M) Consolidated Gross Margin 64.8% Expanded by 180 basis points Cloud Services & AI Revenue $68.4M +28.57% YoY; 72.4% Gross Margin Enterprise Software Revenue $42.6M +9.51% YoY; 78.1% Gross Margin Cybersecurity Revenue $22.5M +13.64% YoY; 58.2% Gross Margin Consulting Services Revenue $15.0M -17.13% YoY; 34.0% Gross Margin Cost of Goods Sold (COGS) $52.27M 35.2% of revenue Research & Development (R&D) $31.20M 21.0% of revenue (increased from $26.5M in Q2) Sales & Marketing (S&M) $35.64M 24.0% of revenue General & Administrative (G&A) $13.36M 9.0% of revenue Operating Income (EBIT) $16.03M 10.8% operating margin Net Income & Diluted EPS $12.82M / $0.41 EPS increased from $0.34 in Q3 FY23 Proposed CapEx Reallocation $3.5M Additional hardware reserve for Q4 multi-tenant API traffic 📊 Operational Key Performance Indicators (KPIs) Operational KPI Value Additional Context Annual Recurring Revenue (ARR) $412.0M Up 18.4% YoY Net Revenue Retention (NRR) 118% Strong cross-selling in enterprise accounts Customer Acquisition Cost (CAC) $14,250 6.8% decrease QoQ CAC Payback Period 11.2 months Improved from 12.8 months in Q2 Customer Lifetime Value (LTV) $92,400 LTV/CAC ratio of 6.48x Enterprise Customer Count (>\$100k ARR) 842 accounts +48 net additions in Q3 Service Level Agreement (SLA) 99.985% Across all core SaaS clusters Employee Retention Rate 94.2% Annualized ⚙️ Project & Risk Metrics Item / Initiative Specific Metric / Detail Project ApexAI Beta Active across 50 partner accounts Cloud Migration Delay Delayed by 2 weeks due to GDPR audit compliance SOC 2 Type II Certification Completed with 0 non-conformities Large Deal Sales Cycles Impacted deals exceeding $250k ARR Inference Optimization Dynamic routing & quantized models reduce inference COGS by 22% 
 
 
 
 
 
 Question 3 
 What is the main purpose or goal of 

📊 Summary: 3 questions answered, 6,297 total tokens used

💡 To ask different questions, modify the 'follow_up_questions' list in this cell and run again!


---

# 🎉 Summary

Congratulations! You've learned how to interact with the Gemini API using direct HTTP requests.

## What You Accomplished

✅ **Authentication**: Used notebook widgets for secure credential input  
✅ **Model Discovery**: Queried available models from the API  
✅ **Simple Requests**: Made single-turn API calls with custom parameters  
✅ **Example 2 - Interactive Chatbot**: Built a multi-turn conversation that maintains context across 4 turns  
✅ **Example 3 - Document Understanding**: Connected to S3, extracted text from .docx using standard library (zipfile + XML), and analyzed documents with Gemini  

## Key Takeaways

1. **Direct HTTP Approach**: Using `requests.post()` gives you full control over API interactions
2. **Message Structure**: Conversations are arrays of `{"role": "user/assistant", "content": "text"}`
3. **Context Management**: Include full conversation history in each request for context-aware responses
4. **Response Path**: Extract responses via `response_data['choices'][0]['message']['content']`
5. **Document Processing**: Word .docx files are ZIP archives - you can parse them with standard library (`zipfile` + `xml.etree.ElementTree`) without external dependencies

## Next Steps

* **Experiment with Parameters**: Try different `temperature` values (0.0 to 1.0) to see how creativity changes
* **Try Different Models**: Switch between flash and pro models to compare performance
* **Build Your Application**: Use the interactive conversation pattern (Example 2) as a foundation for chatbots
* **Add Error Handling**: Enhance the examples with retry logic, rate limiting, and better error messages
* **Stream Responses**: Explore streaming API endpoints for real-time response generation

## 📚 Additional Resources

* [Gemini API Documentation](https://ai.google.dev/docs)
* [OpenAI API Compatibility Guide](https://platform.openai.com/docs/api-reference)

---